In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import syllables 

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) 
nltk.download('stopwords', quiet=True)

print("1. Loading datasets with original index values...")
df = pd.read_csv('spotify_millsongdata.csv')

df['original_index'] = df.index 

target_artists = {
    'Pop': [
        'Rihanna', 'Maroon 5', 'Katy Perry', 'Justin Timberlake', 'Britney Spears', 
        'Taylor Swift', 'Kelly Clarkson', 'Mariah Carey', 'Bruno Mars', 'Madonna', 
        'Michael Jackson', 'Lady Gaga', 'ABBA', 'Adele', 'Ed Sheeran', 
        'Elton John', 'Celine Dion', 'Cher', 'Ariana Grande', 'Selena Gomez'
    ],
    'Rock': [
        'The Beatles', 'Rolling Stones', 'U2', 'Queen', 'Pink Floyd', 
        'Led Zeppelin', 'Nirvana', 'Fleetwood Mac', 'Metallica', "Guns N' Roses", 
        'Aerosmith', 'Bon Jovi', 'Bruce Springsteen', 'David Bowie', 'Dire Straits', 
        'Eagles', 'Genesis', 'Iron Maiden', 'The Who', 'AC/DC'
    ],
    'Rap': [
        'Eminem', 'Lil Wayne', 'Drake', 'Snoop Dogg', 'Nicki Minaj', 
        'Kanye West', 'Jay-Z', '50 Cent', 'Ice Cube', 'Nas', 
        'Wu-Tang Clan', 'Outkast', 'Busta Rhymes', 'Kendrick Lamar', 'Tupac', 
        '2Pac', 'Dr. Dre', 'J. Cole', 'Travis Scott', 'Post Malone'
    ],
    'Country': [
        'Dolly Parton', 'Johnny Cash', 'Willie Nelson', 'George Strait', 'Garth Brooks', 
        'Kenny Rogers', 'Loretta Lynn', 'Hank Williams', 'Tim McGraw', 'Reba McEntire', 
        'Shania Twain', 'Alan Jackson', 'Carrie Underwood', 'Brad Paisley', 'Dixie Chicks', 
        'Faith Hill', 'John Denver', 'Patsy Cline', 'Toby Keith', 'Waylon Jennings'
    ],
    'RnB': [
        'Stevie Wonder', 'Whitney Houston', 'Prince', 'Ray Charles', 'Aretha Franklin', 
        'Marvin Gaye', 'Alicia Keys', 'John Legend', 'Boyz II Men', 'TLC', 
        'Mary J. Blige', 'Chaka Khan', 'Donna Summer', 'Earth, Wind & Fire', 'Gladys Knight', 
        'Lionel Richie', 'Usher', 'Diana Ross', 'Beyonce', "Destiny's Child"
    ]
}

print("2. Filtering...")

filtered_dfs = []
for genre, artists in target_artists.items():
    for artist in artists:
        artist_songs = df[df['artist'] == artist].copy()
        if len(artist_songs) > 0:
            artist_songs['genre'] = genre 
            filtered_dfs.append(artist_songs)

df_subset = pd.concat(filtered_dfs, ignore_index=True)

def clean_and_tokenize(text):
    text = re.sub(r'\[.*?\]|\(.*?\)', '', str(text)) 
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    return [w for w in tokens if not w in stop_words]

def calculate_chorus_ratio(text):
    lines = [line.strip() for line in str(text).lower().split('\n') if line.strip()]
    if not lines:
        return 0.0
    line_counts = pd.Series(lines).value_counts()
    repetitive_lines = line_counts[line_counts > 1].sum()
    return round(repetitive_lines / len(lines), 4)

def calculate_syllable_density(tokens):
    if not tokens:
        return 0.0
    total_syllables = sum(syllables.estimate(word) for word in tokens)
    return round(total_syllables / len(tokens), 4)

final_data = []
genre_counters = {g: 1 for g in target_artists.keys()}

for index, row in df_subset.iterrows():
    raw_text = row['text']
    genre = row['genre']
    artist = row['artist']
    song_name = row['song']
    orig_index = row['original_index']
    
    c_ratio = calculate_chorus_ratio(raw_text)
    
    if c_ratio > 0.0:
        clean_tokens = clean_and_tokenize(raw_text)
        
        if len(clean_tokens) > 0:
            s_density = calculate_syllable_density(clean_tokens)
            
            track_id = f"{genre}_{genre_counters[genre]:05d}"
            genre_counters[genre] += 1
            
            final_data.append({
                'original_index': orig_index,
                'track_id': track_id,
                'song_name': song_name,
                'artist': artist,
                'genre': genre,
                'chorus_ratio': c_ratio,
                'syllable_density': s_density,
                'sentiment_score': 0.0,
                'flesch_kincaid_readability': 0.0,
                'lsa_component_1': 0.0,
                'lsa_component_2': 0.0,
                'lsa_component_3': 0.0
            })

df_final_features = pd.DataFrame(final_data)

print(f"\nRemaining number of songs after filtering: {len(df_final_features)}")

df_final_features.to_csv('billboard_lyrics_features_final.csv', index=False)
print("\nFile has been created.")

1. Loading datasets with original index values...
2. Filtering...
